# Week 2 — Day 2: Text Feature Engineering

## AI-Based Marketplace Analyzer

### Goal
Extract useful numerical and behavioral features from review text.

### Input
Cleaned marketplace dataset from Day 1.

### Output
Review-level text features for later NLP and risk analysis.

In [1]:
import pandas as pd
import numpy as np
import re
import string

## 1. Load Cleaned Dataset

We use the cleaned dataset produced during Day 1 instead of modifying the original raw dataset.

In [2]:
from pathlib import Path
import pandas as pd

project_root = Path.cwd()
if not (project_root / 'data').exists():
    project_root = project_root.parent

df = pd.read_parquet(project_root / 'data' / 'processed' / 'marketplace_cleaned.parquet')
print('Loaded cleaned dataset for feature engineering. Shape:', df.shape)


In [3]:
df.shape

(19991, 22)

In [4]:
df.head()

,rating,title_review,text,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,main_category,...,rating_number,price,store,details,review_year,review_month,review_day,review_day_of_week,review_hour,price_available
0,5.0,Neon pink.,"Nice formula, smooth application with no pooli...",B00JQRPB9I,B00JQRPB9I,AFH5GZTYBKKGZ7SB45FUFLNWMUFQ,2015-05-09 22:04:41.000,4,True,All Beauty,...,18,16.45,China Glaze,"{""Brand"": ""China Glaze"", ""Item Form"": ""Gel"", ""...",2015,5,9,5,22,True
1,5.0,Godd experience,Great product. It came when expected.,B00QHB48M2,B00QHB48M2,AFGLKF5GMQ3CYUBB2KFCODKBOEYA,2017-06-01 01:04:59.000,0,True,All Beauty,...,325,NaN,Max Factor,"{""Item Form"": ""Stick"", ""Color"": ""Beige"", ""Pack...",2017,6,1,3,1,False
2,5.0,Love This..made my mom's day,My 90 year old mother's favorite fragrance.,B01IAFGDYS,B01IAFGDYS,AFTDHU6IXXYGCEJ6JUAI7LFD3JMA,2019-06-24 16:14:04.340,0,True,All Beauty,...,7,NaN,Five Star,"{""Brand"": ""Five Star"", ""Item Form"": ""Spray"", ""...",2019,6,24,0,16,False
3,1.0,Review,Just not worth it,B01DX9HUP2,B01DX9HUP2,AGIEEQAKFGKN7JH4LQKRWTCCMXGA,2017-06-15 21:07:17.278,0,True,All Beauty,...,239,NaN,Buytra,"{""Brand"": ""Buytra"", ""Recommended Uses For Prod...",2017,6,15,3,21,False
4,5.0,Battery will not stay charged (Rescinding),The Battery life on this product has always be...,B07W11NLFS,B07W11NLFS,AFNEEMO6DSJJTMBCIF3SCXUQBH2A,2020-01-27 14:58:37.959,1,True,All Beauty,...,574,NaN,Duomishu,"{""Brand"": ""Duomishu"", ""Power Source"": ""Battery...",2020,1,27,0,14,False


In [5]:
df[["title_review", "text"]].head()

,title_review,text
0,Neon pink.,"Nice formula, smooth application with no pooli..."
1,Godd experience,Great product. It came when expected.
2,Love This..made my mom's day,My 90 year old mother's favorite fragrance.
3,Review,Just not worth it
4,Battery will not stay charged (Rescinding),The Battery life on this product has always be...


In [6]:
df[["title_review", "text"]].dtypes

title_review    str
text            str
dtype: object

## 2. Combine Review Title and Review Text

The review title and review body both contain potentially useful textual information.
We combine them into one analysis field while preserving the original columns.

In [7]:
df["review_text"] = (
    df["title_review"].fillna("").astype(str).str.strip()
    + " "
    + df["text"].fillna("").astype(str).str.strip()
).str.strip()

In [8]:
df[["title_review", "text", "review_text"]].head()

,title_review,text,review_text
0,Neon pink.,"Nice formula, smooth application with no pooli...","Neon pink. Nice formula, smooth application wi..."
1,Godd experience,Great product. It came when expected.,Godd experience Great product. It came when e...
2,Love This..made my mom's day,My 90 year old mother's favorite fragrance.,Love This..made my mom's day My 90 year old mo...
3,Review,Just not worth it,Review Just not worth it
4,Battery will not stay charged (Rescinding),The Battery life on this product has always be...,Battery will not stay charged (Rescinding) The...


## 3. Basic Text Features

### Word Count

Word count measures the number of words contained in each review.

In [9]:
df["review_word_count"] = (
    df["review_text"]
    .str.split()
    .str.len()
)

In [10]:
df["review_word_count"].describe()

count    19991.000000
mean        36.616878
std         46.566122
min          1.000000
25%         11.000000
50%         23.000000
75%         45.000000
max        996.000000
Name: review_word_count, dtype: float64

In [11]:
df["review_char_count"] = df["review_text"].str.len()

In [12]:
df["review_char_count"].describe()

count    19991.000000
mean       194.809964
std        250.325135
min          3.000000
25%         60.000000
50%        124.000000
75%        237.000000
max       5728.000000
Name: review_char_count, dtype: float64

## 4. Punctuation Features

Punctuation can provide useful behavioral information about review writing.
We count punctuation marks while keeping the original review text unchanged.

In [13]:
df["punctuation_count"] = df["review_text"].apply(
    lambda x: sum(char in string.punctuation for char in x)
)

In [14]:
df["punctuation_count"].describe()

count    19991.000000
mean         6.089790
std         10.845054
min          0.000000
25%          1.000000
50%          3.000000
75%          7.000000
max        320.000000
Name: punctuation_count, dtype: float64

In [15]:
df["exclamation_count"] = df["review_text"].str.count("!")

In [16]:
df["exclamation_count"].describe()

count    19991.000000
mean         0.772948
std          1.677059
min          0.000000
25%          0.000000
50%          0.000000
75%          1.000000
max         68.000000
Name: exclamation_count, dtype: float64

In [17]:
df["question_count"] = df["review_text"].str.count(r"\?")

In [18]:
df["question_count"].describe()

count    19991.000000
mean         0.027062
std          0.260100
min          0.000000
25%          0.000000
50%          0.000000
75%          0.000000
max         10.000000
Name: question_count, dtype: float64

## 5. Capitalization Features

We measure uppercase characters in each review.
High capitalization can be useful as a text-style signal, while the original text is preserved.

In [19]:
df["uppercase_count"] = df["review_text"].apply(
    lambda x: sum(char.isupper() for char in x)
)

In [20]:
df["uppercase_ratio"] = df.apply(
    lambda row: (
        row["uppercase_count"] / row["review_char_count"]
        if row["review_char_count"] > 0
        else 0
    ),
    axis=1
)

In [21]:
df["uppercase_ratio"].describe()

count    19991.000000
mean         0.047962
std          0.060135
min          0.000000
25%          0.022936
50%          0.033333
75%          0.052632
max          0.904762
Name: uppercase_ratio, dtype: float64

In [22]:
df[
    [
        "review_text",
        "review_word_count",
        "review_char_count",
        "punctuation_count",
        "exclamation_count",
        "question_count",
        "uppercase_count",
        "uppercase_ratio"
    ]
].head(10)

,review_text,review_word_count,review_char_count,punctuation_count,exclamation_count,question_count,uppercase_count,uppercase_ratio
0,"Neon pink. Nice formula, smooth application wi...",52,274,13,0,0,6,0.021898
1,Godd experience Great product. It came when e...,8,54,2,0,0,3,0.055556
2,Love This..made my mom's day My 90 year old mo...,12,72,5,0,0,3,0.041667
3,Review Just not worth it,5,24,0,0,0,2,0.083333
4,Battery will not stay charged (Rescinding) The...,72,354,7,0,0,6,0.016949
5,There are better options. When it came to me o...,48,250,6,0,0,6,0.024000
6,Good for testing child Purpose: teething child...,36,233,17,0,0,5,0.021459
7,Look nice but don’t stay on. They look nice bu...,22,114,4,0,0,3,0.026316
8,Good Good,2,9,0,0,0,2,0.222222
9,Works great Works great,4,23,0,0,0,2,0.086957


In [23]:
text_features = [
    "review_text",
    "review_word_count",
    "review_char_count",
    "punctuation_count",
    "exclamation_count",
    "question_count",
    "uppercase_count",
    "uppercase_ratio"
]

df[text_features].isna().sum()

review_text          0
review_word_count    0
review_char_count    0
punctuation_count    0
exclamation_count    0
question_count       0
uppercase_count      0
uppercase_ratio      0
dtype: int64

In [24]:
df[
    [
        "review_word_count",
        "review_char_count",
        "punctuation_count",
        "exclamation_count",
        "question_count",
        "uppercase_count",
        "uppercase_ratio"
    ]
].describe()

,review_word_count,review_char_count,punctuation_count,exclamation_count,question_count,uppercase_count,uppercase_ratio
count,19991.000000,19991.000000,19991.000000,19991.000000,19991.000000,19991.000000,19991.000000
mean,36.616878,194.809964,6.089790,0.772948,0.027062,6.442999,0.047962
std,46.566122,250.325135,10.845054,1.677059,0.260100,10.936400,0.060135
min,1.000000,3.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,11.000000,60.000000,1.000000,0.000000,0.000000,3.000000,0.022936
50%,23.000000,124.000000,3.000000,0.000000,0.000000,4.000000,0.033333
75%,45.000000,237.000000,7.000000,1.000000,0.000000,7.000000,0.052632
max,996.000000,5728.000000,320.000000,68.000000,10.000000,457.000000,0.904762


In [25]:
df["uppercase_ratio"].min(), df["uppercase_ratio"].max()

(np.float64(0.0), np.float64(0.9047619047619048))

In [26]:
df[
    [
        "review_text",
        "review_word_count",
        "review_char_count",
        "punctuation_count",
        "exclamation_count",
        "question_count",
        "uppercase_count",
        "uppercase_ratio"
    ]
].sample(10, random_state=42)

,review_text,review_word_count,review_char_count,punctuation_count,exclamation_count,question_count,uppercase_count,uppercase_ratio
8715,Good set of brushes I ordered these brushes to...,102,564,11,1,0,11,0.019504
6463,"Five Stars great price! Double thickness, keep...",17,93,3,1,0,3,0.032258
1815,Nails Awesome recommend 100%,4,28,1,0,0,2,0.071429
9119,"Nice cream, light scent, gentle feel. It's too...",142,737,34,0,0,13,0.017639
12598,Chemical Burn Alert Smells like petroleum and ...,25,148,5,0,0,7,0.047297
19876,17yo daughter approved My 17 yo daughter reall...,10,58,0,0,0,1,0.017241
4356,... because I've been hearing how boar bristle...,86,464,20,0,0,7,0.015086
10261,Terrible Diffuser This diffuser was supposed t...,31,194,6,0,0,6,0.030928
13164,Fast shipping and just as described These are ...,20,119,2,0,0,2,0.016807
16285,Five Stars finally a painless way to brush my ...,13,71,0,0,0,2,0.028169


In [27]:
df[
    [
        "review_word_count",
        "review_char_count",
        "punctuation_count",
        "exclamation_count",
        "question_count",
        "uppercase_count",
        "uppercase_ratio"
    ]
].corr()

,review_word_count,review_char_count,punctuation_count,exclamation_count,question_count,uppercase_count,uppercase_ratio
review_word_count,1.000000,0.996259,0.862632,0.192063,0.171378,0.540723,-0.196773
review_char_count,0.996259,1.000000,0.882932,0.195048,0.177120,0.546277,-0.192684
punctuation_count,0.862632,0.882932,1.000000,0.261723,0.227327,0.529459,-0.115526
exclamation_count,0.192063,0.195048,0.261723,1.000000,0.072117,0.224925,0.026191
question_count,0.171378,0.177120,0.227327,0.072117,1.000000,0.130637,-0.009852
uppercase_count,0.540723,0.546277,0.529459,0.224925,0.130637,1.000000,0.370419
uppercase_ratio,-0.196773,-0.192684,-0.115526,0.026191,-0.009852,0.370419,1.000000


## 7. Save Text-Enhanced Dataset

The cleaned dataset is now enriched with review-level text features.
This dataset will be used as the input for the next stage of Week 2.

In [28]:
df.to_parquet(
    "../data/processed/marketplace_text_features.parquet",
    index=False
)

In [29]:
import os

os.path.exists(
    "../data/processed/marketplace_text_features.parquet"
)

True

In [30]:
import os
import pandas as pd

output_path = "../data/processed/marketplace_text_features.parquet"

print("File exists:", os.path.exists(output_path))

if os.path.exists(output_path):
    saved_df = pd.read_parquet(output_path)

    print("Dataset shape:", saved_df.shape)
    print("Columns:", saved_df.columns.tolist())
    print("Duplicate rows:", saved_df.duplicated().sum())
    display(saved_df.head())

File exists: True
Dataset shape: (19991, 30)
Columns: ['rating', 'title_review', 'text', 'asin', 'parent_asin', 'user_id', 'timestamp', 'helpful_vote', 'verified_purchase', 'main_category', 'title_product', 'average_rating', 'rating_number', 'price', 'store', 'details', 'review_year', 'review_month', 'review_day', 'review_day_of_week', 'review_hour', 'price_available', 'review_text', 'review_word_count', 'review_char_count', 'punctuation_count', 'exclamation_count', 'question_count', 'uppercase_count', 'uppercase_ratio']
Duplicate rows: 0


,rating,title_review,text,asin,parent_asin,user_id,timestamp,helpful_vote,verified_purchase,main_category,...,review_hour,price_available,review_text,review_word_count,review_char_count,punctuation_count,exclamation_count,question_count,uppercase_count,uppercase_ratio
0,5.0,Neon pink.,"Nice formula, smooth application with no pooli...",B00JQRPB9I,B00JQRPB9I,AFH5GZTYBKKGZ7SB45FUFLNWMUFQ,2015-05-09 22:04:41.000,4,True,All Beauty,...,22,True,"Neon pink. Nice formula, smooth application wi...",52,274,13,0,0,6,0.021898
1,5.0,Godd experience,Great product. It came when expected.,B00QHB48M2,B00QHB48M2,AFGLKF5GMQ3CYUBB2KFCODKBOEYA,2017-06-01 01:04:59.000,0,True,All Beauty,...,1,False,Godd experience Great product. It came when e...,8,54,2,0,0,3,0.055556
2,5.0,Love This..made my mom's day,My 90 year old mother's favorite fragrance.,B01IAFGDYS,B01IAFGDYS,AFTDHU6IXXYGCEJ6JUAI7LFD3JMA,2019-06-24 16:14:04.340,0,True,All Beauty,...,16,False,Love This..made my mom's day My 90 year old mo...,12,72,5,0,0,3,0.041667
3,1.0,Review,Just not worth it,B01DX9HUP2,B01DX9HUP2,AGIEEQAKFGKN7JH4LQKRWTCCMXGA,2017-06-15 21:07:17.278,0,True,All Beauty,...,21,False,Review Just not worth it,5,24,0,0,0,2,0.083333
4,5.0,Battery will not stay charged (Rescinding),The Battery life on this product has always be...,B07W11NLFS,B07W11NLFS,AFNEEMO6DSJJTMBCIF3SCXUQBH2A,2020-01-27 14:58:37.959,1,True,All Beauty,...,14,False,Battery will not stay charged (Rescinding) The...,72,354,7,0,0,6,0.016949


## Final Validation and Dataset Status

**Output dataset:** `data/processed/marketplace_text_features.parquet`

**Dataset shape:** 19,991 rows × 30 columns

**Exact duplicate rows:** 0

### Validation results
- The processed Parquet file loads successfully.
- `price_available` matches the presence or absence of `price`.
- `review_word_count` matches the word count calculated from `review_text`.
- `review_char_count` matches the character count of `review_text`.
- No negative review word counts were found.
- All `uppercase_ratio` values are within the expected range of 0 to 1.

### Important limitation
The `price` column contains missing values. Price-based analysis must use available prices and must not treat missing prices as zero.

### Next stage
Proceed to NLP analysis, including review similarity, sentiment analysis, and investigation of unusual review patterns. These signals are indicators for further investigation, not proof of fraudulent activity.